# Tutorial 01: Agents with Tools - Adding Real-World Capabilities

##  Learning Objectives
After completing this notebook, you will be able to:
- Understand what tools are and why they matter
- Create Python functions that agents can call
- Use type annotations and docstrings for tool definitions
- Add tools to agents at creation time and at runtime
- Understand how agents decide when to use tools

##  Key Concepts

### What Are Tools?
Tools are **Python functions** that agents can call to:
- Retrieve real-time information (weather, news, prices)
- Perform calculations (currency conversion, distance)
- Access external APIs (flights, hotels, restaurants)
- Interact with databases and files

### Function Calling Flow
```
User: "What's the weather in Paris?"
  ↓
Agent: "I need to call the weather tool"
  ↓
Tool: get_weather("Paris") → "Sunny, 22°C"
  ↓
Agent: "The weather in Paris is sunny with a high of 22°C"
```

---

## Step 1: Setup

Import what we need. We also include Pydantic for better type hints.

In [ ]:
import asyncio
import os
from datetime import datetime
from random import randint, choice
from typing import Annotated

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from agent_framework.azure import AzureOpenAIChatClient
from pydantic import Field
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# === Authentication Method Selection ===
# True: API Key Authentication, False: DefaultAzureCredential Authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
    print("🔑 Authentication method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    # The framework automatically reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credentials, so we explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())
    print("🔐 Authentication method: DefaultAzureCredential")

print("Imports successful!")

## Step 2: Create Your First Tool

Let's create a simple weather tool. Pay attention to the key elements:
1. **Docstring** - Describes what the function does (the agent sees this!)
2. **Type annotations** - Define parameter types
3. **Field descriptions** - Provide context to the agent
4. **Return type** - What the function returns

In [ ]:
def get_weather(
    location: Annotated[
        str,
        Field(
            description="The city/location to get weather for (e.g., 'Paris', 'Tokyo', 'New York')."
        ),
    ],
) -> str:
    """
    Get the current weather for a specified location.
    Returns the temperature (Celsius) and weather summary.
    """
    # In a real app, you would call a weather API
    # Here we simulate it for simplicity
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    temp = randint(10, 35)
    condition = choice(conditions)

    return f"The weather in {location} is {condition} with a temperature of {temp}°C."


# Test the function directly
print(get_weather("Paris"))

## Step 3: Create an Agent with Tools

Now let's give our Travel Assistant the ability to check weather!

In [ ]:
# Create an agent with the weather tool
travel_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="TravelAssistant",
    instructions="""
    You are a helpful travel assistant. You can recommend destinations, provide travel tips,
    and check the current weather.

    When users ask about weather, use the get_weather tool to provide accurate information.
    """,
    tools=[get_weather],  # Add the tool here!
)

print("Agent created with weather tool!")

## Step 4: Test Tool Usage

Let's ask about weather and watch the agent use the tool.

In [ ]:
query = "What's the weather like in Barcelona? I'm considering a visit."

print(f"User: {query}\n")
response = await travel_agent.run(query)
print(f"Agent: {response}")

 **Notice**: The agent automatically called `get_weather("Barcelona")` and incorporated the result into its response!

## Step 5: Add More Tools

Let's add a currency converter to help with travel budget planning.

In [ ]:
def convert_currency(
    amount: Annotated[float, Field(description="The amount to convert")],
    from_currency: Annotated[
        str, Field(description="Source currency code (e.g., 'USD', 'EUR', 'GBP')")
    ],
    to_currency: Annotated[
        str, Field(description="Target currency code (e.g., 'USD', 'EUR', 'GBP')")
    ],
) -> str:
    """
    Convert an amount from one currency to another.
    Uses approximate exchange rates.
    """
    # Simplified exchange rates (in a real app, use a real-time API)
    rates = {
        "USD": 1.0,
        "EUR": 0.92,
        "GBP": 0.79,
        "JPY": 149.50,
        "CAD": 1.36,
        "AUD": 1.53,
    }

    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    if from_currency not in rates or to_currency not in rates:
        return f"Sorry, exchange rate not available for {from_currency} or {to_currency}."

    # Convert to USD first, then to target currency
    usd_amount = amount / rates[from_currency]
    converted = usd_amount * rates[to_currency]

    return f"{amount} {from_currency} is approximately {converted:.2f} {to_currency}"


# Test
print(convert_currency(100, "USD", "EUR"))

## Step 6: Add a Flight Search Tool

Let's simulate a flight search (in a real app, you would call APIs like Amadeus or Skyscanner).

In [ ]:
def search_flights(
    origin: Annotated[str, Field(description="Departure city or airport code")],
    destination: Annotated[str, Field(description="Destination city or airport code")],
    date: Annotated[str, Field(description="Travel date in YYYY-MM-DD format")],
) -> str:
    """
    Search for available flights between two locations on a given date.
    Returns a summary of flights with prices.
    """
    # Simulate flight search results
    airlines = [
        "United",
        "Delta",
        "American",
        "Air France",
        "British Airways",
        "Lufthansa",
    ]

    flights = []
    for i in range(3):
        airline = choice(airlines)
        price = randint(200, 1200)
        duration = randint(2, 15)
        flights.append(f"- {airline}: ${price} ({duration}h {randint(0, 59)}m)")

    result = f"Found {len(flights)} flights from {origin} to {destination} on {date}:\n"
    result += "\n".join(flights)
    return result


# Test
print(search_flights("New York", "London", "2025-11-15"))

## Step 7: Create an Enhanced Travel Agent

Now let's create an agent with all the tools!

In [ ]:
# Create a more capable agent
enhanced_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="EnhancedTravelAssistant",
    instructions="""
    You are an advanced travel assistant with access to real-time tools.

    You can:
    - Check current weather
    - Convert currencies for budget planning
    - Search for available flights

    When asked for practical travel information, use the appropriate tools.
    Combine tool results with your knowledge to provide comprehensive advice.
    """,
    tools=[get_weather, convert_currency, search_flights],
)

print("Agent created with 3 tools!")

## Step 8: Test Multi-Tool Queries

Let's ask questions that require multiple tools.

In [ ]:
# Query that requires weather and currency conversion
query = "I'm planning a trip to Tokyo. What's the weather like? Also, how much is 500 USD in Japanese yen?"

print(f"User: {query}\n")
response = await enhanced_agent.run(query)
print(f"Agent: {response}")

In [ ]:
# Query that requires all tools
query = """
I want to fly from San Francisco to Paris on December 1, 2025.
Show me available flights, tell me the local weather, and convert 1000 USD to euros.
"""

print(f"User: {query}\n")
response = await enhanced_agent.run(query)
print(f"Agent: {response}")

##  Understanding Tool Execution

Here's what happens behind the scenes:

1. **Agent analyzes the query** - Determines which tools are needed
2. **Tool calls are made** - Functions are executed with extracted parameters
3. **Results are collected** - Tool outputs are gathered
4. **Agent synthesizes response** - Combines tool results with context

The agent can call:
- **No tools** - If the query doesn't require them
- **One tool** - For simple requests
- **Multiple tools** - Sequentially or in parallel
- **The same tool multiple times** - If needed (e.g., weather for multiple cities)

## Step 9: Runtime Tools

You can also add tools to a specific query, not the agent itself.

In [ ]:
# Create an agent without tools
basic_agent = Agent(
    client=AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=AZURE_OPENAI_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
    ),
    name="BasicAgent",
    instructions="You are a helpful travel assistant.",
    # No tools defined here!
)

# Add a tool only for this query
query = "What's the weather in Rome?"
response = await basic_agent.run(query, tools=[get_weather])

print(f"User: {query}")
print(f"Agent: {response}")

## Step 10: Tools with Error Handling

Let's create a more robust tool that handles errors gracefully.

In [ ]:
def get_travel_advisory(
    country: Annotated[str, Field(description="The country to check travel advisories for")],
) -> str:
    """
    Get the latest travel advisory level for a specified country.
    Returns safety information and recommendations.
    """
    try:
        # Simulate advisory data
        advisories = {
            "France": "Level 1: Exercise Normal Precautions",
            "Japan": "Level 1: Exercise Normal Precautions",
            "Mexico": "Level 2: Exercise Increased Caution",
            "Egypt": "Level 2: Exercise Increased Caution",
            "Afghanistan": "Level 4: Do Not Travel",
        }

        country = country.strip().title()

        if country in advisories:
            return f"Travel advisory for {country}: {advisories[country]}"
        else:
            return (
                f"No specific travel advisory found for {country}. "
                "Please check official sources."
            )

    except Exception as e:
        return f"Error retrieving travel advisory: {str(e)}"


# Test
print(get_travel_advisory("Japan"))
print(get_travel_advisory("Atlantis"))  # Fictional country

##  Tool Best Practices

1. **Clear docstrings** - The agent uses these to understand when to call the tool
2. **Type annotations** - Help the agent extract correct parameters
3. **Field descriptions** - Provide examples and context
4. **Error handling** - Return useful messages instead of crashing
5. **Consistent return values** - Always return strings (or use structured output)
6. **Descriptive names** - Function names should clearly indicate their purpose

###  Bad Tool
```python
def weather(x):
    return api.call(x)
```

###  Good Tool
```python
def get_current_weather(
    location: Annotated[str, Field(description="City name or coordinates")],
) -> str:
    """Get real-time weather for a location."""
    try:
        # Implementation
        pass
    except Exception as e:
        return f"Error: {e}"
```

##  Practice Exercises

Try creating these tools:

1. **Hotel Search Tool** - Search hotels in a city by price range
2. **Distance Calculator** - Calculate distance between two cities
3. **Restaurant Finder** - Search restaurants by cuisine type at a location
4. **Timezone Converter** - Convert time between different timezones

In [ ]:
# Exercise 1: Create a hotel search tool
def search_hotels(
    city: Annotated[str, Field(description="The city to search hotels in")],
    max_price: Annotated[int, Field(description="Maximum price per night in USD")],
) -> str:
    """
    Search for hotels in a specified city within budget.
    """
    # Write your implementation here
    pass


# Test your tool!

##  Key Takeaways

1. **Tools extend agent capabilities** beyond just the LLM's knowledge
2. **Agents make smart decisions** about when and how to use tools
3. **Function signatures matter** - good annotations = better tool usage
4. **Multiple tools can work together** to solve complex queries
5. **Tools can be added** at agent creation time or per query

##  What's Still Missing

Our agent still:
-  Doesn't remember previous conversations
-  Can't maintain context between queries
-  Starts fresh with each `run()` call

**In the next tutorial**, we'll add **Threads** for stateful multi-turn conversations!

---